# k-means embedding

`2. US_checkin_v2`에 저장된 `loc_cluster_id_bestk`를 바탕으로 사용자별 클러스터 방문 분포를 dense embedding으로 만들고 MongoDB에 저장합니다.

In [11]:
from datetime import datetime, timezone

import numpy as np
from pymongo import MongoClient, UpdateOne
from tqdm.auto import tqdm


In [12]:
HOST = "10.255.68.40"
PORT = 27017
DB_NAME = "ejoow"

CHECKIN_COL = "2. US_checkin_v2"
CLUSTER_INFO_COL = "3. loc_cluster_centroids_bestk"
META_COL = "3. loc_kmeans_bestk_meta"
OUT_COL = "4. user_cluster_embedding_bestk"
ASSIGN_FIELD = "loc_cluster_id_bestk"
NORMALIZE = "logprob"
BULK_SIZE = 5000

client = MongoClient(host=HOST, port=PORT)
db = client[DB_NAME]
checkin_col = db[CHECKIN_COL]
cluster_info_col = db[CLUSTER_INFO_COL]
meta_col = db[META_COL]
out_col = db[OUT_COL]

print("db:", DB_NAME)
print("checkin input:", CHECKIN_COL)
print("cluster info input:", CLUSTER_INFO_COL)
print("meta input:", META_COL)
print("embedding output:", OUT_COL)


db: ejoow
checkin input: 2. US_checkin_v2
cluster info input: 3. loc_cluster_centroids_bestk
meta input: 3. loc_kmeans_bestk_meta
embedding output: 4. user_cluster_embedding_bestk


## 1. 클러스터 정보 확인

In [13]:
meta_doc = meta_col.find_one(sort=[("updated_at", -1)])
if meta_doc is None:
    raise ValueError("bestk 메타 문서가 없습니다. k-means4.ipynb 메타 저장 셀을 먼저 실행하세요.")

K = int(meta_doc["k"])
cluster_docs = list(
    cluster_info_col.find(
        {"k": K},
        {"_id": 0, "cluster_id": 1, "latitude": 1, "longitude": 1}
    ).sort("cluster_id", 1)
)

if not cluster_docs:
    meta_centroids = meta_doc.get("centroids", [])
    cluster_docs = [
        {
            "cluster_id": int(cluster_id),
            "latitude": float(latitude),
            "longitude": float(longitude),
        }
        for cluster_id, (latitude, longitude) in enumerate(meta_centroids)
    ]
    print("centroid collection is empty; using centroids from meta document")

cluster_ids = [int(doc["cluster_id"]) for doc in cluster_docs]
print("cluster_ids:", cluster_ids)
print("num_clusters:", len(cluster_ids))

if len(cluster_docs) != K:
    raise ValueError(f"클러스터 정보가 {K}개가 아닙니다. 현재: {len(cluster_docs)}")

print("best_k:", K)
cluster_docs


centroid collection is empty; using centroids from meta document
cluster_ids: [0, 1, 2, 3, 4]
num_clusters: 5
best_k: 5


[{'cluster_id': 0,
  'latitude': 40.82884979248047,
  'longitude': -73.99425506591797},
 {'cluster_id': 1,
  'latitude': 40.825687408447266,
  'longitude': -73.99960327148438},
 {'cluster_id': 2,
  'latitude': 40.654666900634766,
  'longitude': -74.00653076171875},
 {'cluster_id': 3,
  'latitude': 40.772438049316406,
  'longitude': -73.8897933959961},
 {'cluster_id': 4,
  'latitude': 40.660621643066406,
  'longitude': -73.78041076660156}]

## 2. 사용자별 클러스터 방문 빈도 집계

In [14]:
pipeline = [
    {"$match": {ASSIGN_FIELD: {"$exists": True}}},
    {
        "$group": {
            "_id": {"user_id": "$user_id", "cluster_id": f"${ASSIGN_FIELD}"},
            "count": {"$sum": 1},
        }
    },
    {
        "$group": {
            "_id": "$_id.user_id",
            "total_checkins": {"$sum": "$count"},
            "pairs": {
                "$push": {
                    "k": {"$toString": "$_id.cluster_id"},
                    "v": "$count"
                }
            },
        }
    },
    {
        "$project": {
            "_id": 0,
            "user_id": "$_id",
            "total_checkins": 1,
            "frequency": {"$arrayToObject": "$pairs"},
        }
    },
]

user_cluster_rows = list(checkin_col.aggregate(pipeline, allowDiskUse=True))
print("users:", len(user_cluster_rows))
user_cluster_rows[:2]


users: 67207


[{'total_checkins': 3, 'user_id': '1445323', 'frequency': {'1': 1, '0': 2}},
 {'total_checkins': 2, 'user_id': '1311548', 'frequency': {'2': 1, '1': 1}}]

## 3. dense embedding 생성 함수

In [15]:
def build_cluster_embedding(freq_dict, k, normalize="prob"):
    vec = np.zeros(k, dtype=np.float32)

    for cluster_id_str, count in freq_dict.items():
        cluster_id = int(cluster_id_str)
        if 0 <= cluster_id < k:
            vec[cluster_id] = float(count)

    if normalize == "prob":
        total = vec.sum()
        if total > 0:
            vec /= total
    elif normalize == "logprob":
        vec = np.log1p(vec)
        total = vec.sum()
        if total > 0:
            vec /= total

    return vec


## 4. 사용자 임베딩 저장

In [16]:
out_col.create_index("user_id", unique=True)

ops = []
written = 0
now = datetime.now(timezone.utc)

for row in tqdm(user_cluster_rows, desc="Writing user cluster embeddings"):
    user_id = row["user_id"]
    freq = row.get("frequency", {})
    vec = build_cluster_embedding(freq, k=K, normalize=NORMALIZE)

    out_doc = {
        "_id": user_id,
        "user_id": user_id,
        "k": int(K),
        "assign_field": ASSIGN_FIELD,
        "source_collection": CHECKIN_COL,
        "cluster_info_collection": CLUSTER_INFO_COL,
        "normalize": NORMALIZE,
        "total_checkins": int(row.get("total_checkins", 0)),
        "nonzero": int((vec > 0).sum()),
        "frequency": freq,
        "embedding": vec.tolist(),
        "updated_at": now,
    }

    ops.append(UpdateOne({"_id": user_id}, {"$set": out_doc}, upsert=True))

    if len(ops) >= BULK_SIZE:
        out_col.bulk_write(ops, ordered=False)
        written += len(ops)
        ops.clear()

if ops:
    out_col.bulk_write(ops, ordered=False)
    written += len(ops)

print("written:", written)
print("out count:", out_col.estimated_document_count())


Writing user cluster embeddings:   0%|          | 0/67207 [00:00<?, ?it/s]

written: 67207
out count: 67207


## 5. 저장 결과 확인

In [17]:
sample = out_col.find_one({}, {"_id": 1, "user_id": 1, "k": 1, "total_checkins": 1, "nonzero": 1, "embedding": 1})
sample


{'_id': '1445323',
 'embedding': [0.6131471991539001, 0.38685280084609985, 0.0, 0.0, 0.0],
 'k': 5,
 'nonzero': 2,
 'total_checkins': 3,
 'user_id': '1445323'}